## Multi-Agent Trip Planner

In [ ]:
import json
from datetime import date
from typing import Annotated, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
import os
os.environ["OPENAI_API_KEY"] = ""

# ── Constants ──────────────────────────────────────────────────────────────────

REQUIRED_FIELDS = [
    "travel_type",           # who is travelling?
    "num_travelers",         # how many?
    "source_location",       # starting from?
    "destination",           # going to?
    "travel_dates",          # when?
    "places_to_visit",       # specific spots?
    "transportation_preferences",  # how to get there?
    "food_preferences",      # any dietary needs?
    "trip_budget_luxury",    # comfort level?
    "budget",                # total budget?
]

SYSTEM_PROMPT = """
You are a warm, friendly travel planner assistant. Today's date is {today}.
Collect trip details one question at a time in a natural, conversational way.

────────────────────────────────────────────────
QUESTION ORDER & HOW TO ASK
────────────────────────────────────────────────
Ask fields in this order, skipping already-collected ones:

  1. travel_type       — "Are you travelling Solo, as a Couple, with Family, or with Friends/Group?"
  2. num_travelers     — Ask based on travel_type context:
                         Solo → confirm it's just them (num=1)
                         Couple → confirm 2, or ask if more
                         Family/Friends → "How many people in total, including yourself?"
  3. source_location   — "Which city or town are you starting from?"
  4. destination       — "Where are you heading? (city, region, or country)"
  5. travel_dates      — "When are you planning to travel?"
                         Accept ANY natural language: "next Monday to Friday",
                         "last week of June", "26th to 30th July", "next weekend" etc.
                         Resolve relative dates using today's date and store as "YYYY-MM-DD to YYYY-MM-DD".
  6. places_to_visit   — "Any specific places, landmarks, or attractions you'd like to cover?"
                         These can be anywhere within the COUNTRY of the destination — not limited to the city.
                         Example: destination = Mumbai → places can include Goa, Pune, Ajanta Caves (all within India).
                         Only flag if a place is in a completely different country.
  7. transportation_preferences — "How would you prefer to travel? (Flight / Train / Bus / Road trip / Mixed)"
  8. food_preferences  — "Any food preferences or dietary needs? (Veg / Non-veg / Vegan / Halal / No preference)"
  9. trip_budget_luxury — "What's your comfort level? (Budget / Standard / Luxury / Ultra-Luxury)"
  10. budget           — "What's your total budget for this trip? (please include currency, e.g. ₹50,000 or $2,000)"

────────────────────────────────────────────────
INLINE VALIDATION (reject and re-ask immediately)
────────────────────────────────────────────────
  ✗ source_location == destination → "Looks like source and destination are the same — where are you heading?"
  ✗ num_travelers ≤ 0 → "That doesn't seem right — how many people are travelling?"
  ✗ travel_dates fully in the past → "Those dates have already passed — when are you planning to go?"
  ✗ budget < ~₹500 / $10 for any trip → "That budget seems too low for travel — could you share a realistic number?"
  Keep all rejection messages to one friendly sentence.

────────────────────────────────────────────────
HOLISTIC VALIDATION (run once ALL fields are collected)
────────────────────────────────────────────────
Check and add a short note to validation_issues for each problem found:

  • Budget reality    — Is budget reasonable for destination + duration + num_travelers + luxury tier?
                        e.g. "₹2,000 for 7 nights in Paris for 3 people is unrealistic."
  • Luxury mismatch  — Does trip_budget_luxury match the budget?
                        e.g. budget = ₹5,000 total but luxury = Ultra-Luxury → flag.
  • Geographic outlier — Are places_to_visit within the destination COUNTRY?
                        Flag only if a place is clearly close to each other in the geogrphical location or cluster, anything out of the geographical bounds → flag.
                        e.g. list of all destinations = India, places include "Tokyo" → flag.
                        e.g. list of all destinations = Kerala, places include "Ayodhya" → flag.
  • Travel type mismatch — Does travel_type match num_travelers?
                        e.g. travel_type = Solo but num_travelers = 4 → flag.
  • Duration check   — Is the trip long enough to cover all listed places?

If issues found → status = "validating", list issues briefly, ask user to confirm or correct.
If all clear    → status = "confirmed", say: "All good — ready to plan your trip! ✈️"

────────────────────────────────────────────────
RE-COLLECTION (if user fixes something)
────────────────────────────────────────────────
- Ask only for the fields that need correction.
- Re-run holistic validation after each correction.
- Repeat until confirmed.

RULES:
- One short message at a time — never ask two questions together.
- Sound like a helpful friend, not a form.
- Accept casual / shorthand answers and extract the right value.
- Only put newly provided/updated values in field_updates.
- Never echo back all collected data unless the user asks.
- Don't ask the questions repeatedly if the correct answer is already provided
"""

# ── State ──────────────────────────────────────────────────────────────────────
class TravelPlannerState(TypedDict):
    messages: Annotated[list, add_messages]
    travel_data: dict
    status: str
    issues: list[str]
    #──────────────────────────────────────────────────────
    weather_agent_data: Optional[dict]  # output from weather agent
    accommodation_agent_data: Optional[dict]  # output from accommodation agent
    transportation_agent_data: Optional[dict]  # output from transportation agent
    hotel_agent_data: Optional[dict]  # output from hotel agent
    places_explorer_agent_data: Optional[dict]  # output from places explorer agent
    budget_agent_data: Optional[dict]  # output from budget agent
    itinerary_agent_data: Optional[dict]  # output from itinerary agent
    final_plan_agent_data: Optional[dict]  # output from final plan agent

# ── Structured LLM Output ──────────────────────────────────────────────────────
class transportation_agent_agent(TravelPlannerState: travel_data):
    transport_data = travel_data.get("transportation_preferences", None)

class TravelDataUpdate(BaseModel):
    """Only fields the user just provided — leave others as None."""
    source_location: Optional[str] = None
    destination: Optional[str] = None
    travel_dates: Optional[str] = None
    num_travelers: Optional[int] = None
    budget: Optional[str] = None
    travel_type: Optional[str] = None
    food_preferences: Optional[str] = None
    transportation_preferences: Optional[str] = None
    places_to_visit: Optional[list[str]] = None
    trip_budget_luxury: Optional[str] = None


class AgentDecision(BaseModel):
    message_to_user: str = Field(
        description="Friendly message, question, or validation feedback to show the user"
    )
    field_updates: TravelDataUpdate = Field(
        default_factory=TravelDataUpdate,
        description="Fields extracted from user's latest reply — only set the ones just provided"
    )
    validation_issues: list[str] = Field(
        default_factory=list,
        description="Brief list of holistic issues found (empty if all good)"
    )
    status: str = Field(
        description="Current stage: 'collecting' while gathering fields, "
                    "'validating' when holistic issues exist, 'confirmed' when all clear"
    )



# ── LLM setup ─────────────────────────────────────────────────────────────────



from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key= os.getenv("OPENAI_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

structured_llm = llm.with_structured_output(AgentDecision)


# ── Nodes ──────────────────────────────────────────────────────────────────────

def agent_node(state: TravelPlannerState) -> dict:
    current_data = state.get("travel_data", {})
    missing = [f for f in REQUIRED_FIELDS if f not in current_data]

    context = SystemMessage(content=f"""
{SYSTEM_PROMPT.format(today=date.today().strftime("%A, %d %B %Y"))}

--- CURRENT COLLECTED DATA ---
{json.dumps(current_data, indent=2)}

--- MISSING FIELDS ---
{missing if missing else "None — all fields collected, run holistic validation now."}
""")

    response: AgentDecision = structured_llm.invoke(
        [context] + state["messages"]
    )

    new_fields = {k: v for k, v in response.field_updates.model_dump().items() if v is not None}
    updated_data = {**current_data, **new_fields}

    return {
        "messages": [AIMessage(content=response.message_to_user)],
        "travel_data": updated_data,
        "status": response.status,
        "issues": response.validation_issues,
    }


def human_node(state: TravelPlannerState) -> dict:
    last = state["messages"][-1]
    print(f"\nPlanner: {last.content}\n")
    user_input = input("You: ").strip()
    return {"messages": [HumanMessage(content=user_input)]}


# ── Routing ────────────────────────────────────────────────────────────────────

def route(state: TravelPlannerState) -> str:
    if state.get("status") == "confirmed":
        return END
    return "human"


# ── Graph ──────────────────────────────────────────────────────────────────────

graph = (
    StateGraph(TravelPlannerState)
    .add_node("agent", agent_node)
    .add_node("human", human_node)
    .add_edge(START, "agent")
    .add_conditional_edges("agent", route, {"human": "human", END: END})
    .add_edge("human", "agent")
    .compile()
)


# ── Entry point ────────────────────────────────────────────────────────────────

def run_input_agent() -> dict:
    """
    Runs the interactive travel input agent.
    Returns the final collected travel_data dict for downstream agents.
    """
    print("\n" + "=" * 50)
    print("   TRAVEL PLANNER — Input Collection Agent")
    print("=" * 50)

    final_state = graph.invoke(
        {
            "messages": [HumanMessage(content="Hi, I want to plan a trip.")],
            "travel_data": {},
            "status": "collecting",
            "issues": [],
        },
        config={"recursion_limit": 150},
    )

    print("\n" + "=" * 50)
    print("Input collection complete. Handing off to planner agents.")
    print("=" * 50)
    print(f"Final state: {final_state}\n")
    return final_state["travel_data"]


if __name__ == "__main__":
    travel_data = run_input_agent()
    print("\nFinal collected data:")
    print(json.dumps(travel_data, indent=2))



   TRAVEL PLANNER — Input Collection Agent
{'messages': [AIMessage(content='That sounds exciting! Are you travelling Solo, as a Couple, with Family, or with Friends/Group?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'travel_data': {}, 'status': 'collecting', 'issues': []}

Planner: That sounds exciting! Are you travelling Solo, as a Couple, with Family, or with Friends/Group?

{'messages': [AIMessage(content="Just to confirm, it's just you travelling, right?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'travel_data': {'num_travelers': 1, 'travel_type': 'Solo'}, 'status': 'collecting', 'issues': []}

Planner: Just to confirm, it's just you travelling, right?

{'messages': [AIMessage(content='Great! Which city or town are you starting from?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'travel_data': {'num_travelers': 1, 'travel_type': 'Solo'}, 'status': 'collecting